# N014 · 均摊复杂度与操作计数

**目标：** 区分单次昂贵操作和整个序列的总成本。

**先修：** N009, N011。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 聚合法；记账法；势能直觉；动态数组；每元素至多进出一次。

## 从问题到算法

两个栈实现队列时不能每次出队都倒腾全部元素。仅当输出栈为空才把输入栈整体反转：最早入队的元素出现在输出栈顶。单次转移可能线性，但每个元素一生最多转移一次，所以长操作序列均摊常数。

## 最小实现

**本章接口：** `TwoStackQueue`、`instrumented_push_pop(ops)`

In [1]:
class TwoStackQueue:
    def __init__(self):
        self.incoming=[]
        self.outgoing=[]
        self.transfers=0
    def push(self,x):
        self.incoming.append(x)
    def pop(self):
        if not self.outgoing:
            while self.incoming:
                self.outgoing.append(self.incoming.pop())
                self.transfers+=1
        if not self.outgoing:
            raise IndexError('pop from empty queue')
        return self.outgoing.pop()
    def empty(self):
        return not (self.incoming or self.outgoing)

def instrumented_push_pop(ops):
    queue=TwoStackQueue()
    popped=[]
    trace=[]
    for op,value in ops:
        if op=='push': queue.push(value)
        elif op=='pop': popped.append(queue.pop())
        else: raise ValueError('unknown operation')
        trace.append((op,list(queue.incoming),list(queue.outgoing),queue.transfers))
    return popped,queue.transfers,trace

## 正确性、前提与复杂度

逻辑队列顺序始终是 reversed(outgoing) 接 incoming。一次批量反转后顺序不变。给每次push预付其未来迁移成本，总迁移次数不超过push次数。

**代价：** m次合法操作总 O(m)，单次pop最坏 O(n)、均摊 O(1)；存储 O(n)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

_,_,trace=instrumented_push_pop([('push',1),('push',2),('pop',None),('push',3),('pop',None),('pop',None)])
show_table(['操作','输入栈','输出栈','累计迁移'],trace)

操作,输入栈,输出栈,累计迁移
push,[1],[],0
push,"[1, 2]",[],0
pop,[],[2],2
push,[3],[2],2
pop,[3],[],2
pop,[],[],3


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
a,moves,_=instrumented_push_pop([('push',1),('push',2),('push',3),('pop',None),('pop',None),('pop',None)])
assert a==[1,2,3] and moves==3
q=TwoStackQueue()
for i in range(20):
    q.push(i); assert q.pop()==i
assert q.transfers==20 and q.empty()
print('N014: 所有本章断言通过')

N014: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 构造某次出队很慢的序列。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 证明一批操作总成本线性。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 232. Implement Queue using Stacks（preview）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。